# Image similarity

Run after `python -m similarity.images download` and `python -m similarity.images embed`.
The photos live outside the repo, in `IMAGE_DIR`.

In [ ]:
# Work from the repo root, so `similarity` imports and the data/ paths resolve when opened from notebooks/
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import similarity.search as s
from similarity.embeddings import rescale
from similarity.images import IMAGE_DIR, photo_path
from similarity.preprocessing import load_products

products = load_products()
f = s.load_features()
img = s.image_embeddings(f)
img.model, img.vectors.shape, int(img.present.sum()), round(img.baseline, 3)

In [ ]:
s.METHOD_WEIGHTS

## photos

In [ ]:
missing = products.loc[~img.present, ["uniq_id", "name", "image_url"]]
len(missing), missing["image_url"].isna().sum()

In [ ]:
pd.read_csv(IMAGE_DIR / "failed.csv")["error"].value_counts()

In [ ]:
def show(rows, titles=None, size=2.2):
    fig, axes = plt.subplots(1, len(rows), figsize=(size * len(rows), size * 1.35))
    for i, (ax, row) in enumerate(zip(np.atleast_1d(axes), rows)):
        path = photo_path(IMAGE_DIR, products.loc[row, "uniq_id"])
        if path.exists():
            ax.imshow(Image.open(path))
        ax.set_title((titles[i] if titles else "") + "\n" + products.loc[row, "name"][:28], fontsize=7)
        ax.axis("off")
    plt.tight_layout()

show([0, 1, 2, 3, 4, 5])

## random pairs: why rescale

In [ ]:
rng = np.random.default_rng(0)
both = np.flatnonzero(img.present)
a, b = rng.choice(both, 5000), rng.choice(both, 5000)
raw = np.einsum("ij,ij->i", img.vectors[a], img.vectors[b])
spread = pd.DataFrame({"clip": raw, "clip_rescaled": rescale(raw, img.baseline)})
spread.describe(percentiles=[0.5, 0.9, 0.99]).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
for ax, column in zip(axes, spread.columns):
    ax.hist(spread[column], bins=50, range=(0, 1))
    ax.set_title(column)
axes[0].axvline(img.baseline, color="black", linestyle="--")
axes[0].set_ylabel("random pairs")
plt.tight_layout()

## nearest photos (photo only)

In [ ]:
def nearest(uniq_id, weights, k=5):
    row = f.row_of[uniq_id]
    order, score = s.rank(f, row, weights)
    show([row, *order[:k]], ["query", *[f"{x:.2f}" for x in score[:k]]])

nearest("26d41bdc1495de290bc8e6062d927729", {"image": 1.0})   # Q13 kalamkari blouse fabric

In [ ]:
nearest("45a59081eda442a2d35efb62b936e3b1", {"image": 1.0})   # Q19 "OOMPH!"

In [ ]:
nearest("2304753b72be21b48128f7d2ae482c9f", {"image": 1.0})   # Q14 baby frock

## one product, four methods

In [ ]:
def compare(uniq_id, k=10):
    row = f.row_of[uniq_id]
    print(products.loc[row, "name"])
    tops = {}
    for method, weights in s.METHOD_WEIGHTS.items():
        order, _ = s.rank(f, row, weights)
        tops[method] = products.loc[order[:k], "name"].str[:38].tolist()
    return pd.DataFrame(tops, index=range(1, k + 1))

In [ ]:
compare("26d41bdc1495de290bc8e6062d927729")   # Q13 kalamkari blouse fabric

In [ ]:
compare("45a59081eda442a2d35efb62b936e3b1")   # Q19 "OOMPH!"

In [ ]:
row = f.row_of["26d41bdc1495de290bc8e6062d927729"]
for method in ["tfidf", "tfidf_image"]:
    order, score = s.rank(f, row, s.METHOD_WEIGHTS[method])
    show([row, *order[:5]], ["query", *[f"{x:.2f}" for x in score[:5]]])

## breakdown

In [ ]:
order, _ = s.rank(f, row, s.METHOD_WEIGHTS["tfidf_image"])
s.score_breakdown(f, row, order[1], s.METHOD_WEIGHTS["tfidf_image"]).round(3)

## no photo: the photo score is skipped

In [ ]:
no_photo = int(np.flatnonzero(~img.present)[0])
products.loc[no_photo, ["name", "image_url"]]

In [ ]:
order, _ = s.rank(f, no_photo, s.METHOD_WEIGHTS["tfidf_image"])
s.score_breakdown(f, no_photo, order[0], s.METHOD_WEIGHTS["tfidf_image"]).round(3)

In [ ]:
text_only, _ = s.rank(f, no_photo, s.METHOD_WEIGHTS["tfidf"])
with_image, _ = s.rank(f, no_photo, s.METHOD_WEIGHTS["tfidf_image"])
(text_only[:10] == with_image[:10]).all()

## what CLIP sees (its text side)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(img.model, device="cpu")
kinds = ["a saree", "a kurta", "a t-shirt", "a shirt", "a dress", "a cap", "fabric", "briefs"]
k = model.encode(kinds, normalize_embeddings=True)
rows = [f.row_of[u] for u in ["26d41bdc1495de290bc8e6062d927729", "45a59081eda442a2d35efb62b936e3b1",
                              "2304753b72be21b48128f7d2ae482c9f", "981b995e1e8016ef7c17b98b782aa88f"]]
pd.DataFrame(img.vectors[rows] @ k.T, columns=kinds, index=products.loc[rows, "name"].str[:40]).round(2)

## timing

In [ ]:
for method, weights in s.METHOD_WEIGHTS.items():
    start = time.perf_counter()
    for r in range(100):
        s.rank(f, r, weights)
    print(f"{method:13s} {(time.perf_counter() - start) / 100 * 1000:.1f} ms per search")

## evaluation

`python -m evals.run_eval --alternative minilm_image` writes the workbook.

In [ ]:
pd.read_excel("evals/eval_results.xlsx", sheet_name="Methods")